# S.C.R.A.M v2: a bigger, newer dataset for the art detector (Kaggle)

v1 called two Gemini images human-made. That had two causes, and v2 fixes both:

1. **No modern generators in the data.** v1's AI images came from older models (Stable Diffusion 1.x, Latent Diffusion and similar). v2 adds Google's Imagen 3/4 and Nano Banana (Gemini's image model), GPT Image 1, Midjourney v6/v7, DALL·E 3, Flux and SDXL.
2. **Training and testing didn't see images the same way.** v1 shrank every training image to 512 px and re-saved it as a JPEG, but the test cell gave the model the full-size PNG. v2 prepares your test images exactly like the training images, and stores that setting in the model file.

**Datasets** (about 120,000 images with the defaults, vs about 35,000 in v1)

| | Human-made | AI-generated |
|---|---|---|
| From v1 | WikiArt, Modern Art, Human-AI Artwork, Real & Fake Art, AI-ArtBench (human halves) | Human-AI Artwork, Real & Fake Art, AI-ArtBench |
| From `georgeblck/art-datasets` | **Best Artworks of All Time**, **Art Museums PD 440K** (Met, Smithsonian, Art Institute of Chicago, Cleveland) | **DiffusionDB** (Stable Diffusion art prompts), **Pick-a-Pic** (SDXL, Dreamlike) |
| Modern generators | OpenFake real photos, AI-vs-Real photos | **OpenFake** (Imagen 3/4, Nano Banana, GPT Image 1, Midjourney v6/v7, DALL·E 3, Flux…), **Nano Banana Pro** set, AI-vs-Real (Midjourney, DALL·E, SD) |

**Before you start (Kaggle)**

1. *Session options → Accelerator → GPU T4 x2* (or P100). Training uses one GPU, so the second T4 sits idle.
2. *Session options → Internet → On.* Needed for GitHub and Hugging Face. Kaggle asks you to verify your phone number once.
3. **Add the Kaggle datasets as inputs** (*+ Add Input*, search each name): `steubk/wikiart`, `amansunesh/modern-art-dataset`, `ikarus777/best-artworks-of-all-time`, `adamelkholy/human-ai-artwork`, `doctorstrange420/real-and-fake-ai-generated-art-images-dataset`, `ravidussilva/real-ai-art`, `tristanzhang32/ai-generated-images-vs-real-images`. The notebook reads them straight from `/kaggle/input`, with no download. Any it doesn't find, it downloads through the Kaggle API (WikiArt alone is about 27 GB), which needs the secrets in the next step.
4. 🔑 *Add-ons → Secrets* (only if some dataset isn't attached): `KAGGLE_USERNAME` and `KAGGLE_KEY`. `HF_TOKEN` (huggingface.co → Settings → Access Tokens, read access) is optional, but it makes the Hugging Face downloads more reliable. Turn each secret on for this notebook.
5. **Use Save Version → Save & Run All (Commit), not the interactive Run All.** A commit run goes on in the background, so a closed browser tab or an idle timeout can't kill it.

**Time and space.** Preparing the data takes about 1-1.5 h, and training on a T4 about 2.5-3.5 h, which fits in Kaggle's 12 h session limit. The images are prepared in scratch space (`/tmp`), not in `/kaggle/working`, so they don't count against the 20 GB output limit. Only the model, metrics and plots are kept in the output.

**If the session dies mid-training:** run *Save Version* when you can, then attach that version's output to a new notebook (*+ Add Input → Your Work*). The run folder is found there and training resumes from the last finished epoch. (Set `SAVE_CACHE` to also keep the prepared dataset, if it fits in 20 GB.)

## 1. Settings

In [ ]:
#@title Settings { display-mode: "form" }
OUT_DIR = "/kaggle/working/scram_art"  #@param {type:"string"}
GITHUB_URL = "https://github.com/mru-art/AI-detector-model"  #@param {type:"string"}
#@markdown Save each prepared dataset as a zip in the output (about 10 GB of the 20 GB limit), so a restart skips the data preparation.
SAVE_CACHE = True  #@param {type:"boolean"}

#@markdown **Human-made art**
USE_WIKIART = True  #@param {type:"boolean"}
USE_MODERN_ART = True  #@param {type:"boolean"}
USE_BEST_ARTWORKS = True  #@param {type:"boolean"}
USE_MUSEUMS_PD = True  #@param {type:"boolean"}
#@markdown **AI art, older generators**
USE_HUMAN_AI_ARTWORK = True  #@param {type:"boolean"}
USE_REAL_FAKE_ART = True  #@param {type:"boolean"}
USE_AI_ARTBENCH = True  #@param {type:"boolean"}
USE_DIFFUSIONDB = True  #@param {type:"boolean"}
USE_PICKAPIC = True  #@param {type:"boolean"}
#@markdown **Modern generators (Gemini/Imagen, GPT Image, Midjourney v6+, Flux), with real photos to balance them**
USE_OPENFAKE = True  #@param {type:"boolean"}
USE_NANO_BANANA_PRO = True  #@param {type:"boolean"}
USE_AI_VS_REAL = True  #@param {type:"boolean"}
#@markdown Other Kaggle image datasets, comma-separated `owner/slug`. Add `:human` if every image in it is human-made.
EXTRA_KAGGLE = ""  #@param {type:"string"}

#@markdown **Size**: images per class from each dataset, and per generator from OpenFake
MAX_PER_CLASS = 6000  #@param {type:"integer"}
OPENFAKE_PER_GENERATOR = 1200  #@param {type:"integer"}

#@markdown **Training**
TASK = "binary"  #@param ["binary", "multiclass"]
EPOCHS = 8  #@param {type:"integer"}
BATCH_SIZE = 32  #@param {type:"integer"}
#@markdown Keep on: every image (training, test, and later in the app) is shrunk to this size and saved as a JPEG first,
#@markdown so the model can't tell human from AI by file type or resolution.
HARMONIZE = True  #@param {type:"boolean"}
MAX_SIDE = 512  #@param {type:"integer"}
#@markdown Small sample and 1 epoch, to check everything works (~30 min). Results go to separate folders.
QUICK_TEST = False  #@param {type:"boolean"}
SEED = 42

In [ ]:
#@title Paths and helpers (no need to change) { display-mode: "form" }
import os, sys, io, re, csv, json, glob, random, shutil, hashlib, zipfile, tarfile, threading, subprocess, posixpath
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

IN_KAGGLE = os.path.exists("/kaggle/working")
KAGGLE_INPUT = Path("/kaggle/input")
# Scratch space outside /kaggle/working: that folder is saved as the notebook's output (20 GB limit).
WORK = Path("/tmp/scram_work") if IN_KAGGLE else Path(os.environ.get("SCRAM_WORK", "kaggle_work")).resolve()
CODE = WORK / "scram"
VIT = CODE / "vision_transformer"
DATA = WORK / "dataset"               # train/ validation/ test/ <class>/
META = DATA / "_meta"                 # one manifest per dataset
RAW = WORK / "raw"                    # downloads (deleted after use)
OUT = Path(OUT_DIR)                   # kept as the notebook's output
CACHE_REL = "cache_quicktest" if QUICK_TEST else "cache"
CACHE = OUT / CACHE_REL
JPEG_QUALITY = 95

if QUICK_TEST:
    MAX_PER_CLASS, OPENFAKE_PER_GENERATOR, EPOCHS = min(MAX_PER_CLASS, 300), min(OPENFAKE_PER_GENERATOR, 40), 1

def sig_of(*parts):
    return hashlib.sha1(json.dumps(list(parts)).encode()).hexdigest()[:6]

def sh(cmd, cwd=None):
    """Run a command and stream its output into the notebook."""
    print("$", " ".join(map(str, cmd)), flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TQDM_MININTERVAL": "20"}
    p = subprocess.Popen(list(map(str, cmd)), cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    while chunk := p.stdout.read1(4096):
        sys.stdout.write(chunk.decode(errors="replace")); sys.stdout.flush()
    if p.wait():
        raise RuntimeError(f"Command failed (exit code {p.returncode}). See the output above.")

def secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient          # Kaggle: Add-ons -> Secrets
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None


def find_prior(rel, depth=4):
    """A file or folder from an earlier run: in the output folder, or in an attached notebook output
    (Kaggle mounts those as /kaggle/input/<notebook>/...)."""
    p = OUT / rel
    if p.exists():
        return p
    for d in range(1, depth + 1):
        hits = sorted(glob.glob(str(KAGGLE_INPUT) + "/*" * d + "/" + rel))
        if hits:
            return Path(hits[0])
    return None


def harmonize_bytes(data, max_side=MAX_SIDE, quality=JPEG_QUALITY):
    """Shrink so the short side is at most max_side, then save as JPEG. Used for training images,
    for your test images, and (with the backend patch) in the app, so all three match."""
    from PIL import Image, ImageFile, ImageOps
    ImageFile.LOAD_TRUNCATED_IMAGES, Image.MAX_IMAGE_PIXELS = True, None
    with Image.open(io.BytesIO(data)) as im:
        w, h = im.size
        if min(w, h) > max_side:                     # JPEGs decode straight at a smaller size: much faster
            im.draft("RGB", (max(1, w * max_side // min(w, h)), max(1, h * max_side // min(w, h))))
        img = ImageOps.exif_transpose(im).convert("RGB")
    w, h = img.size
    if min(w, h) > max_side:
        s = max_side / min(w, h)
        img = img.resize((max(1, round(w * s)), max(1, round(h * s))), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=quality)
    return buf.getvalue()

print("Working folder:", WORK, "| Output:", OUT)

## 2. GPU, internet, project code and sign-ins

In [ ]:
import torch, socket
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
n_gpu = torch.cuda.device_count()
print("GPU:", GPU or "none", f"(x{n_gpu}; training uses one)" if n_gpu > 1 else "")
if not GPU:
    print("⚠️  No GPU. Session options → Accelerator → GPU T4 x2, then run all the cells again.")

try:
    socket.create_connection(("github.com", 443), timeout=8).close()
except OSError:
    raise RuntimeError("No internet. Session options → Internet → On (Kaggle asks for phone verification once), "
                       "then run all the cells again.")

OUT.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
if not (VIT / "train.py").exists():
    sh(["git", "clone", "--depth", "1", GITHUB_URL, CODE])
sys.path.insert(0, str(VIT))
import art_datasets as ad          # the project's labelling and split helpers

free = shutil.disk_usage(WORK).free / 1e9
print(f"Project: {CODE}\nFree scratch disk: {free:.0f} GB")
if free < 40:
    print("⚠️  Less than 40 GB free. Attach the big Kaggle datasets (WikiArt) as inputs instead of downloading them.")

In [ ]:
# Packages Kaggle doesn't already have (no -U: don't disturb the preinstalled versions)
sh([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "pyarrow",
    "scikit-learn", "seaborn", "tqdm", "tensorboard"])

# Kaggle API: only needed for datasets that are NOT attached as inputs. Secrets first, then ~/.kaggle/kaggle.json.
user, key, token = secret("KAGGLE_USERNAME"), secret("KAGGLE_KEY"), secret("KAGGLE_API_TOKEN")
if user and key:
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = user, key
elif token:
    os.environ["KAGGLE_API_TOKEN"] = token
api = None
try:
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    print("Kaggle API: signed in")
except Exception as e:
    api = None
    print(f"Kaggle API: not signed in ({type(e).__name__}). Fine if every Kaggle dataset is attached as an input; "
          "otherwise add KAGGLE_USERNAME and KAGGLE_KEY under Add-ons → Secrets.")

# Hugging Face: optional token (public datasets work without one, just with stricter rate limits)
HF_TOKEN = secret("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
from huggingface_hub import HfFileSystem, hf_hub_download
hffs = HfFileSystem(token=HF_TOKEN)
print("Hugging Face:", "token found" if HF_TOKEN else "no token (fine; add HF_TOKEN if downloads get rate-limited)")

## 3. Download, write `structure.txt`, label and split

For each dataset this cell:

1. reads it from `/kaggle/input` if you attached it, otherwise downloads it (the big Hugging Face sets are read only in part),
2. writes `structure.txt`, the folder/file listing Pavan's `tree` command makes, to `OUT_DIR/structures/`,
3. labels each image: from folder names for the Kaggle sets (check the table it prints), from the dataset's own `real/fake` and generator columns for the Hugging Face sets,
4. takes up to `MAX_PER_CLASS` images per class and splits them 80/10/10 (an image always lands in the same split),
5. shrinks and re-saves them as JPEGs into `dataset/`, and, if `SAVE_CACHE` is on, caches that dataset's part in the output.

A dataset that fails (renamed, rules not accepted, rate limit) is skipped with a warning, and the rest carry on. Run the cell again later to retry it.

In [ ]:
def K(handle, **kw):  return dict(kind="kaggle", handle=handle, **kw)
def HF(kind, repo, **kw): return dict(kind=kind, handle=repo, **kw)

# rule "human": every image is human-made. rule "auto": the nearest folder name that says ai/fake/generated
# or human/real decides; images in other folders get `unclear` (None = skip).
SOURCES = [
    # ---- human-made
    K("steubk/wikiart", key="wikiart", rule="human", use=USE_WIKIART),
    K("amansunesh/modern-art-dataset", key="modern_art", rule="auto", unclear="human",
      ai_class="ai_modern_art", use=USE_MODERN_ART),
    K("ikarus777/best-artworks-of-all-time", key="best_artworks", rule="human", use=USE_BEST_ARTWORKS,
      skip_paths=["resized"]),                       # the same paintings are in there twice
    HF("hf_tar", "Mitsua/art-museums-pd-440k", key="museums_pd", use=USE_MUSEUMS_PD),
    # ---- AI art, older generators (+ their human halves)
    K("adamelkholy/human-ai-artwork", key="human_ai_artwork", rule="auto", unclear=None,
      ai_class="ai_artwork", use=USE_HUMAN_AI_ARTWORK),
    K("doctorstrange420/real-and-fake-ai-generated-art-images-dataset", key="real_fake_art", rule="auto",
      unclear=None, ai_class="ai_art_realfake", use=USE_REAL_FAKE_ART),
    K("ravidussilva/real-ai-art", key="ai_artbench", rule="auto", unclear="human",
      ai_class="ai_artbench", use=USE_AI_ARTBENCH),
    HF("hf_zips", "poloclub/diffusiondb", key="diffusiondb", ai_class="ai_diffusiondb", use=USE_DIFFUSIONDB,
       files=[f"images/part-{i:06d}.zip" for i in range(1, 2001)]),
    HF("hf_parquet", "yuvalkirstain/pickapic_v1", key="pickapic", ai_class="ai_pickapic", use=USE_PICKAPIC,
       image_col="jpg_0", label_col=None, model_col="model_0", per_model=MAX_PER_CLASS // 3,
       include=["train"], exclude=["validation", "test"]),
    # ---- modern generators, with real photos
    HF("hf_parquet", "ComplexDataLab/OpenFake", key="openfake", ai_class="ai_openfake", use=USE_OPENFAKE,
       image_col="image", label_col="label", model_col="model", per_model=OPENFAKE_PER_GENERATOR,
       model_classes=True, include=["train"],
       exclude=["validation", "test", "reddit", "supplement_edit", "supplement_civitai"]),
    HF("hf_parquet", "julienlucas/midjourney-dalle-sd-nanobananapro-dataset", key="nano_banana_pro",
       ai_class="ai_nbp_mix", use=USE_NANO_BANANA_PRO, image_col="image", label_col="label", model_col=None,
       include=[], exclude=[]),
    K("tristanzhang32/ai-generated-images-vs-real-images", key="ai_vs_real", rule="auto", unclear=None,
      ai_class="ai_vs_real", use=USE_AI_VS_REAL),
]
for item in filter(None, (s.strip() for s in EXTRA_KAGGLE.split(","))):
    handle, _, kind = item.partition(":")
    k = "ext_" + handle.split("/")[-1].replace("-", "_")[:30]
    SOURCES.append(K(handle.strip(), key=k, ai_class="ai_" + k[4:], use=True,
                     rule="human" if kind.strip() == "human" else "auto", unclear=None))


def source_sig(src):
    # Kaggle sets keep v1's fingerprint, so v1's prepared split can be reused
    if src["kind"] == "kaggle":
        return sig_of(MAX_PER_CLASS, HARMONIZE, MAX_SIDE, SEED)
    return sig_of(MAX_PER_CLASS, HARMONIZE, MAX_SIDE, SEED, src["kind"], src.get("per_model"))


def is_image(name):
    base = posixpath.basename(name)
    return (Path(base).suffix.lower() in ad.IMG_EXT | {".webp"} and not base.startswith("._")
            and "__MACOSX" not in name)


def slug(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")[:28] or "unknown"


def save_structure(key, lines):
    out = OUT / "structures" / f"{key}_structure.txt"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text("\n".join(lines))
    print("\n".join(lines[:22]) + ("\n..." if len(lines) > 22 else ""))
    print(f"(full listing: {out})")


def tree_lines(key, names):
    """tree /F /a style listing, kept short: folders with image counts and 3 example files each."""
    total, examples = Counter(), defaultdict(list)
    for n in names:
        parts = n.split("/")[:-1]
        for i in range(len(parts) + 1):
            total[tuple(parts[:i])] += 1
        if len(examples[tuple(parts)]) < 3:
            examples[tuple(parts)].append(posixpath.basename(n))
    lines = [f"{key}: {len(names)} images"]
    for d in sorted(total):
        if d:
            lines.append(f"{'|   ' * (len(d) - 1)}+---{d[-1]}/  ({total[d]} images)")
        lines += [f"{'|   ' * len(d)}{f}" for f in examples.get(d, [])]
    return lines


class Writer:
    """Splits, harmonizes and writes one dataset's images, a few threads at a time."""

    def __init__(self, src):
        self.src, self.rows, self.pending = src, [], []
        self.pool = ThreadPoolExecutor(8)

    def _write(self, get_bytes, dst_stem, ext):
        try:
            data = get_bytes()
            if HARMONIZE:
                data, ext = harmonize_bytes(data), ".jpg"
            dst = dst_stem.with_suffix(ext)
            dst.parent.mkdir(parents=True, exist_ok=True)
            dst.write_bytes(data)
            return dst
        except Exception:
            return None                                   # unreadable image: leave it out

    def add(self, uid, cls, get_bytes, ext=".png"):
        key = self.src["key"]
        split = ad.split_of(f"{key}:{uid}", SEED)          # same image -> same split, every run
        stem = DATA / split / cls / f"{key}_{hashlib.sha1(uid.encode()).hexdigest()[:12]}"
        self.pending.append((split, cls, uid, self.pool.submit(self._write, get_bytes, stem, ext.lower())))
        if len(self.pending) >= 1000:
            self.flush()

    def flush(self):
        for split, cls, uid, job in self.pending:
            dst = job.result()
            if dst:
                self.rows.append((split, cls, dst.name, uid))
        self.pending = []
        print(f"  {len(self.rows)} images written", flush=True)

    def finish(self):
        self.flush()
        self.pool.shutdown()
        if not self.rows:
            raise RuntimeError("no images were written")
        key = self.src["key"]
        with open(META / f"{key}.csv", "w", newline="") as fh:     # also marks this dataset as done
            w = csv.writer(fh)
            w.writerow(["split", "class", "file", "source_id"])
            w.writerows(self.rows)
        counts = Counter((s, c) for s, c, _, _ in self.rows)
        print(f"{'class':<28}{'train':>8}{'val':>8}{'test':>8}")
        for c in sorted({c for _, c in counts}):
            print(f"{c:<28}{counts[('train', c)]:>8}{counts[('validation', c)]:>8}{counts[('test', c)]:>8}")
        save_cache(self.src)


def manifest(key):
    with open(META / f"{key}.csv") as fh:
        return list(csv.DictReader(fh))


def save_cache(src):
    if not SAVE_CACHE:
        return
    CACHE.mkdir(parents=True, exist_ok=True)
    out, tmp = CACHE / f"{src['key']}_{source_sig(src)}.zip", WORK / "cache_tmp.zip"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_STORED) as z:       # JPEGs don't compress further
        z.write(META / f"{src['key']}.csv", f"_meta/{src['key']}.csv")
        for r in manifest(src["key"]):
            z.write(DATA / r["split"] / r["class"] / r["file"], f"{r['split']}/{r['class']}/{r['file']}")
    shutil.copy2(tmp, out); tmp.unlink()
    print(f"Cached: {out.name} ({out.stat().st_size / 1e9:.2f} GB)")


def restore(src):
    """Unpack this dataset from a cache of an earlier run, or from v1's combined split. True if it worked."""
    key, sig = src["key"], source_sig(src)
    cached = find_prior(f"{CACHE_REL}/{key}_{sig}.zip")
    if cached:
        with zipfile.ZipFile(cached) as z:
            z.extractall(DATA)
        return True
    legacy = find_prior(f"art_dataset_split_{sig}.zip") if src["kind"] == "kaggle" else None   # v1: one zip for all
    if legacy:
        with zipfile.ZipFile(legacy) as z:
            if f"_meta/{key}.csv" in z.namelist():
                z.extract(f"_meta/{key}.csv", DATA)
                for r in manifest(key):
                    z.extract(f"{r['split']}/{r['class']}/{r['file']}", DATA)
                print(f"  (reused from v1's {legacy.name})")
                save_cache(src)
                return True
    return False


# ------------------------------------------------------------------ kaggle zips, HF zip parts
def kaggle_zip(handle):
    if api is None:
        raise RuntimeError(f"{handle} isn't attached as an input and the Kaggle API isn't signed in. "
                           "Add it with + Add Input, or add KAGGLE_USERNAME and KAGGLE_KEY under Add-ons → Secrets")
    dest = RAW / handle.replace("/", "__")
    for attempt in (1, 2):
        zips = sorted(dest.glob("*.zip"))
        if not zips:
            dest.mkdir(parents=True, exist_ok=True)
            print(f"Downloading {handle} from Kaggle ...", flush=True)
            api.dataset_download_files(handle, path=str(dest), unzip=False, quiet=False)
            zips = sorted(dest.glob("*.zip"))
        if not zips:
            raise RuntimeError("Kaggle returned no zip")
        try:
            with zipfile.ZipFile(zips[0]) as z:
                z.namelist()
            return zips[0]
        except zipfile.BadZipFile:                       # left over from a disconnect mid-download
            print("Incomplete download, starting again"); shutil.rmtree(dest)
    raise RuntimeError("Download keeps failing")


_local = threading.local()

def zip_reader(path, member):
    def read():
        zs = _local.__dict__.setdefault("zips", {})        # one open zip per thread
        if path not in zs:
            zs[path] = zipfile.ZipFile(path)
        return zs[path].read(member)
    return read


def label_of(src, name):
    if src.get("rule") == "human":
        return "real"
    if src.get("rule") == "ai":
        return src["ai_class"]
    lab = ad.label_from_path(Path(name)) or src.get("unclear")
    return None if lab is None else ("real" if lab == "human" else src["ai_class"])


def import_files(src, entries):
    """entries: list of (name used for labels/ids, function returning the bytes)."""
    names = [n for n, _ in entries]
    save_structure(src["key"], tree_lines(src["key"], names))
    by_class, by_dir = defaultdict(list), Counter()
    for n, get in entries:
        cls = label_of(src, n)
        by_dir[(posixpath.dirname(n), cls or "skipped (unclear)")] += 1
        if cls:
            by_class[cls].append((n, get))
    print(f"\nHow the folders were labelled ({len(by_dir)} folders, largest first):")
    for (d, cls), c in sorted(by_dir.items(), key=lambda kv: -kv[1])[:15]:
        print(f"  {c:>7}  {d or '(root)':<60.60} -> {cls}")
    if len(by_dir) > 15:
        print(f"  ... and {len(by_dir) - 15} smaller folders (see structure.txt)")
    skipped = sum(c for (d, cls), c in by_dir.items() if cls.startswith("skipped"))
    if skipped:
        print(f"⚠️  {skipped} images skipped: their folder names don't say AI or human.")

    rng, w = random.Random(SEED), Writer(src)
    for cls, lst in sorted(by_class.items()):
        lst = sorted(lst, key=lambda e: e[0])
        if len(lst) > MAX_PER_CLASS:
            lst = sorted(rng.sample(lst, MAX_PER_CLASS), key=lambda e: e[0])
        print(f"Writing {len(lst)} '{cls}' images ...", flush=True)
        for n, get in lst:
            w.add(n, cls, get, Path(n).suffix)
    w.finish()


def attached_dir(handle):
    """Folder of a dataset added with '+ Add Input' (Kaggle mounts it under /kaggle/input)."""
    owner, slug = handle.split("/")
    for pat in (slug, f"*/{owner}/{slug}", f"*/{slug}"):
        for p in sorted(KAGGLE_INPUT.glob(pat)):
            if p.is_dir():
                return p
    return None


def file_reader(path):
    return lambda: Path(path).read_bytes()


def import_kaggle(src):
    skip = lambda n: not is_image(n) or any(s in n for s in src.get("skip_paths", []))
    root = attached_dir(src["handle"])
    if root:
        print(f"Using the attached input {root}", flush=True)
        names = [p.relative_to(root).as_posix() for p in root.rglob("*") if p.is_file()]
        names = [n for n in names if not skip(n)]
        if not names:
            raise RuntimeError(f"no images in {root}")
        import_files(src, [(n, file_reader(root / n)) for n in names])
        return
    zpath = kaggle_zip(src["handle"])
    with zipfile.ZipFile(zpath) as z:
        all_names = z.namelist()
    names = [n for n in all_names if not skip(n)]
    if not names:
        kinds = Counter(Path(n).suffix.lower() or "(folder)" for n in all_names)
        raise RuntimeError(f"no images in this dataset. It has: {dict(kinds.most_common(6))}")
    import_files(src, [(n, zip_reader(zpath, n)) for n in names])
    shutil.rmtree(zpath.parent, ignore_errors=True)


def import_hf_zips(src):
    """A few randomly chosen zip parts of a big Hugging Face dataset (DiffusionDB: 1,000 images each)."""
    n_parts = -(-MAX_PER_CLASS // 1000)
    parts = sorted(random.Random(SEED).sample(src["files"], n_parts))
    dest, entries = RAW / src["key"], []
    for f in parts:
        print(f"Downloading {src['handle']}/{f} ...", flush=True)
        p = Path(hf_hub_download(src["handle"], f, repo_type="dataset", local_dir=dest, token=HF_TOKEN))
        with zipfile.ZipFile(p) as z:
            entries += [(f"{Path(f).stem}/{m}", zip_reader(p, m)) for m in z.namelist() if is_image(m)]
    import_files({**src, "rule": "ai"}, entries)              # every image here is AI-generated
    shutil.rmtree(dest, ignore_errors=True)


# ------------------------------------------------------------------ HF parquet (OpenFake, Pick-a-Pic, ...)
def parquet_files(src):
    root = f"datasets/{src['handle']}"
    files = sorted(hffs.glob(f"{root}/**/*.parquet"))
    rel = lambda f: f[len(root) + 1:]
    keep = [f for f in files if not any(x in rel(f) for x in src["exclude"])]
    chosen = [f for f in keep if all(x in rel(f) for x in src["include"])] or keep
    if not chosen:
        raise RuntimeError(f"no parquet files found (repo has {len(files)})")
    return root, chosen


def class_names_of(pf, col):
    """Names of a Hugging Face ClassLabel column, if the label is stored as a number."""
    try:
        meta = json.loads(pf.schema_arrow.metadata[b"huggingface"])
        return meta["info"]["features"][col]["names"]
    except Exception:
        return None


def real_or_fake(v, names):
    if v is None:
        return None
    if isinstance(v, int) and names and 0 <= v < len(names):
        v = names[v]
    s = str(v).strip().lower()
    if s in ("real", "human", "0", "false", "authentic"):
        return "real"
    if s in ("fake", "ai", "1", "true", "generated", "synthetic"):
        return "fake"
    return None


def import_hf_parquet(src):
    import pyarrow.parquet as pq
    root, files = parquet_files(src)
    rng = random.Random(SEED)
    rng.shuffle(files)
    label_col, model_col, image_col = src["label_col"], src["model_col"], src["image_col"]
    per_model = src.get("per_model") or MAX_PER_CLASS
    got, budget = Counter(), 30e9 if not QUICK_TEST else 2e9      # stop after reading this many bytes
    read_bytes, w, lines = 0, Writer(src), [f"{src['key']}: {len(files)} parquet files used, e.g."]
    lines += [f"  {f[len(root) + 1:]}" for f in files[:10]]

    fake_total = MAX_PER_CLASS * 4 if src.get("model_classes") else MAX_PER_CLASS

    def quota_left(lab, model):
        if lab == "real":
            return got["real"] < MAX_PER_CLASS
        return got[("fake", model)] < per_model and got["fake"] < fake_total

    def full():
        return got["fake"] >= fake_total and (not label_col or got["real"] >= MAX_PER_CLASS)

    for f in files:
        if read_bytes > budget or full():
            break
        pf = pq.ParquetFile(hffs.open(f, "rb", block_size=8 * 2**20))
        cols = pf.schema_arrow.names
        if image_col not in cols:
            raise RuntimeError(f"no '{image_col}' column; columns are {cols}")
        names = class_names_of(pf, label_col) if label_col else None
        want = [c for c in (label_col, model_col) if c and c in cols]
        meta = pf.read(columns=want).to_pydict() if want else {}
        start = 0
        groups = []
        for g in range(pf.num_row_groups):
            n = pf.metadata.row_group(g).num_rows
            groups.append((g, start, n)); start += n
        rng.shuffle(groups)
        for g, start, n in groups:
            chosen = []
            for i in range(start, start + n):
                lab = real_or_fake(meta[label_col][i], names) if label_col else "fake"
                model = "real" if lab == "real" else (str(meta[model_col][i]) if model_col in meta else "mixed")
                if lab and quota_left(lab, model):
                    got[lab] += 1
                    if lab == "fake":
                        got[("fake", model)] += 1
                    chosen.append((i, lab, model))
            if not chosen:
                continue
            import pyarrow as pa
            col_arr = pf.read_row_group(g, columns=[image_col]).column(image_col)
            vals = col_arr.take(pa.array([i - start for i, _, _ in chosen])).to_pylist()   # only the kept rows
            del col_arr
            read_bytes += pf.metadata.row_group(g).total_byte_size
            for (i, lab, model), v in zip(chosen, vals):
                data = v.get("bytes") if isinstance(v, dict) else v
                if not data:
                    continue
                cls = "real" if lab == "real" else (
                    "ai_" + slug(model) if src.get("model_classes") else src["ai_class"])
                w.add(f"{f[len(root) + 1:]}#{i}", cls, (lambda d=data: d), ".png")
            w.flush()                                      # keeps memory flat
        print(f"  read {read_bytes / 1e9:.1f} GB so far", flush=True)
    lines.append("Taken: " + ", ".join(f"{k if isinstance(k, str) else k[1]}={v}" for k, v in sorted(got.items(), key=str)))
    save_structure(src["key"], lines)
    w.finish()


# ------------------------------------------------------------------ HF WebDataset tars (Art Museums PD)
def import_hf_tar(src):
    root = f"datasets/{src['handle']}"
    shards = sorted(hffs.glob(f"{root}/**/*.tar"))
    random.Random(SEED).shuffle(shards)
    w, n, lines = Writer(src), 0, [f"{src['key']}: {len(shards)} tar shards, read in this order:"]
    for shard in shards:
        if n >= MAX_PER_CLASS:
            break
        lines.append(f"  {shard[len(root) + 1:]}")
        print(f"Reading {shard[len(root) + 1:]} ...", flush=True)
        with hffs.open(shard, "rb", block_size=16 * 2**20) as fh, tarfile.open(fileobj=fh, mode="r|*") as tf:
            for m in tf:                                   # streamed: stops early, no full download
                if n >= MAX_PER_CLASS:
                    break
                if m.isfile() and is_image(m.name):
                    data = tf.extractfile(m).read()
                    w.add(f"{Path(shard).name}/{m.name}", "real", (lambda d=data: d), Path(m.name).suffix)
                    n += 1
    lines.append(f"Taken: {n} images, all human-made (public-domain museum works)")
    save_structure(src["key"], lines)
    w.finish()


IMPORTERS = {"kaggle": import_kaggle, "hf_zips": import_hf_zips, "hf_parquet": import_hf_parquet,
             "hf_tar": import_hf_tar}

META.mkdir(parents=True, exist_ok=True)
added, failed = [], []
for src in SOURCES:
    if not src["use"]:
        continue
    if (META / f"{src['key']}.csv").exists():
        print(f"✓ {src['key']}: ready")
        continue
    if restore(src):
        print(f"✓ {src['key']}: restored from Drive")
        continue
    print(f"\n==================== {src['key']}  ({src['handle']}) ====================")
    try:
        IMPORTERS[src["kind"]](src)
        added.append(src["key"])
    except Exception as e:
        failed.append(src["key"])
        print(f"⚠️  Skipped {src['handle']}: {type(e).__name__}: {e}")
        if "no images" not in str(e):
            print("    Run this cell again later to retry. For Kaggle datasets: attach them with + Add Input, "
              "or open the dataset page once to accept its rules.")
        for p in DATA.glob(f"*/*/{src['key']}_*"):        # remove a half-written dataset
            p.unlink()
print(f"\nAdded now: {added or 'none'} | skipped: {failed or 'none'} | free disk: {shutil.disk_usage(WORK).free / 1e9:.0f} GB")

## 4. Check the split

In [ ]:
def count(split, cls):
    d = DATA / split / cls
    return sum(1 for _ in d.iterdir()) if d.is_dir() else 0

used = [s["key"] for s in SOURCES if s["use"] and (META / f"{s['key']}.csv").exists()]
# drop datasets that are on disk from an earlier run but switched off now
for s in SOURCES:
    if not s["use"] and (META / f"{s['key']}.csv").exists():
        for p in DATA.glob(f"*/*/{s['key']}_*"):
            p.unlink()
        (META / f"{s['key']}.csv").unlink()
for d in DATA.glob("*/*"):
    if d.is_dir() and d.parent.name in ("train", "validation", "test") and not any(d.iterdir()):
        d.rmdir()

CLASSES = sorted(p.name for p in (DATA / "train").iterdir() if p.is_dir())
real_n = sum(count(s, "real") for s in ("train", "validation", "test"))
ai_n = sum(count(s, c) for s in ("train", "validation", "test") for c in CLASSES if c != "real")
print(f"{'class':<28}{'train':>8}{'val':>8}{'test':>8}")
for c in CLASSES:
    print(f"{c:<28}{count('train', c):>8}{count('validation', c):>8}{count('test', c):>8}")
print(f"\nHuman-made: {real_n} | AI-generated: {ai_n} | total: {real_n + ai_n}")
print("Datasets:", ", ".join(used))
if "real" not in CLASSES or len(CLASSES) < 2:
    raise RuntimeError("The detector needs human art (real) and at least one AI class. See the warnings in section 3.")

RUN_NAME = f"art_{TASK}_v2_{sig_of(sorted((s['key'], source_sig(s)) for s in SOURCES if s['key'] in used))}" + (
    "_quicktest" if QUICK_TEST else "")
RUN = OUT / "runs" / RUN_NAME
print("Run folder:", RUN)

## 5. Train

The project's ViT-B/16 (`vision_transformer/train.py`): 1 epoch with the backbone frozen, then up to `EPOCHS` epochs of full fine-tuning, stopping early when validation macro-F1 stops improving. It's then tested on the held-out test split.

Every epoch is saved to `/kaggle/working`. **If the session dies**, save a version, attach its output to a new notebook (*+ Add Input → Your Work*) and run all the cells again: it finds the run there and resumes from the last finished epoch. Training uses one GPU (the project's `train.py` has no multi-GPU code).

In [ ]:
best, last = RUN / "best_model.pth", RUN / "last.pth"
if not RUN.exists():                                   # resume from an attached output of an earlier session
    prior = find_prior(f"runs/{RUN_NAME}")
    if prior:
        print("Copying the run found in", prior)
        shutil.copytree(prior, RUN)

if (RUN / "test_metrics.json").exists():
    print("Already trained and tested:", RUN)
else:
    cmd = [sys.executable, "train.py", "--data_dir", DATA, "--task", TASK,
           "--output_dir", RUN.parent, "--run_name", RUN_NAME,
           "--epochs_head", 1, "--epochs_fine", EPOCHS, "--batch_size", BATCH_SIZE,
           "--num_workers", 2 if IN_KAGGLE else 0, "--patience", 3, "--seed", SEED]
    if last.exists():
        print("Resuming the run saved in", RUN)
        cmd += ["--resume", RUN]
    n_train = sum(count("train", c) for c in CLASSES)
    speed = {"T4": 110, "L4": 220, "A100": 520}.get(next((k for k in ("A100", "L4", "T4") if GPU and k in GPU), ""), 90)
    print(f"{n_train} training images. Rough time on {GPU}: up to "
          f"{n_train * (EPOCHS + 0.5) * 1.15 / speed / 3600:.1f} h (less if early stopping kicks in)\n")
    sh(cmd, cwd=VIT)

# Record how images must be prepared, so the app (with the backend patch) does exactly the same.
ck = torch.load(best, map_location="cpu", weights_only=True)
if HARMONIZE and ck.get("config", {}).get("harmonize") is None:
    ck.setdefault("config", {})["harmonize"] = {"max_side": MAX_SIDE, "quality": JPEG_QUALITY}
    torch.save(ck, best)
print("\n✅ Model:", best)

## 6. Results

In [ ]:
from IPython.display import Image as Show, Markdown, display

m = json.loads((RUN / "test_metrics.json").read_text())
b = m["real_vs_ai"]
names = {"accuracy": "Accuracy", "precision_ai": "Precision (AI)", "recall_ai": "Recall (AI)", "f1_ai": "F1 (AI)",
         "real_recall": "Human art correctly called human", "roc_auc": "ROC-AUC", "avg_precision": "Average precision",
         "eer": "Equal error rate"}
rows = [(names[k], f"{b[k]:.3f}" if k in ("roc_auc", "avg_precision") else f"{b[k]:.1%}") for k in names if k in b]
display(Markdown("**Human vs AI, test split**\n\n| Metric | Value |\n|---|---|\n"
                 + "\n".join(f"| {a} | {v} |" for a, v in rows)))
display(Markdown("**Accuracy per class** (`real` = correctly called human, the rest = correctly called AI; "
                 "OpenFake has one row per generator)\n\n| Class | Accuracy |\n|---|---|\n"
                 + "\n".join(f"| {k} | {v:.1%} |" for k, v in sorted(b["per_source_accuracy"].items(), key=lambda kv: kv[1]))))
for png in ["training_curves.png", "test_confusion_matrix.png", "test_real_vs_ai_roc.png"]:
    if (RUN / png).exists():
        display(Show(str(RUN / png), width=620))

## 7. Try it on your own images

Put your test images in a folder and set `TEST_DIR`: easiest is *+ Add Input → Upload* (a new dataset with your images, mounted as `/kaggle/input/<name>`), or upload them into `/kaggle/working/test_images` with the right-hand file panel. With no images, this cell does nothing, so Run All still finishes.

Images are prepared exactly like the training images first (shrunk and re-saved as JPEG). That's the step v1 was missing.

In [ ]:
TEST_DIR = "/kaggle/working/test_images"  #@param {type:"string"}

exts = ad.IMG_EXT | {".webp"}
paths = sorted(p for p in Path(TEST_DIR).rglob("*") if p.is_file() and p.suffix.lower() in exts) if Path(TEST_DIR).is_dir() else []
if not paths:
    print(f"No images found in {TEST_DIR}. Add some (see the text above) and run this cell again.")
else:
    os.environ["MODEL_PATH"] = str(RUN / "best_model.pth")
    sys.path.insert(0, str(CODE))
    from backend.inference import Detector
    det = Detector(); det.load()
    if det.info.error:
        raise RuntimeError(det.info.error)
    for p in paths:
        raw = p.read_bytes()
        r = det.analyze(harmonize_bytes(raw) if HARMONIZE else raw)
        verdict = "AI-generated" if r["prediction"] == "AI" else "human-made"
        print(f"\n{p.name}: likely {verdict} ({r['ai_probability']:.0%} AI)")
        for note in r["notes"]:
            print("   note:", note)
        display(Show(str(p), width=300))

## 8. Use the model in the app

When the run finishes, the model is in the notebook's output: `scram_art/runs/<run name>/best_model.pth`, and a copy at `/kaggle/working/best_model.pth`. Open the notebook's **Output** tab after *Save Version* (or the file panel on the right) and download it. Copy it into your project as
`vision_transformer/runs/<any name>/best_model.pth`, or point `MODEL_PATH` at it.

**Apply `harmonize.patch` to the project first.** Without it the backend analyses uploads at full size, and you get v1's wrong answers on Gemini images again. The model file records how to prepare images (`config.harmonize`); the patch makes the backend and `predict.py` follow it.

In [ ]:
shutil.copy2(RUN / "best_model.pth", "/kaggle/working/best_model.pth")     # easy to find in the Output tab
print("Model:         ", RUN / "best_model.pth", "(copy: /kaggle/working/best_model.pth)")
print("Test metrics:  ", RUN / "test_metrics.json")
print("structure.txt: ", OUT / "structures")
if SAVE_CACHE:
    print("Dataset caches:", CACHE)